# P5b: Colab T4 コンパイル確認 / P5c: 別承認後の実行
まず **コンパイル確認まで** 実行します。P5c の実行セルは別の承認後に実行してください。
入力束は gmail の Drive `MyDrive/viveMonte/p5/p5b/`。追加パッケージのインストールは不要です。
Claude が bundle を作成・照合した後、下の固定コミットと2個の SHA-256 を引き渡し値で埋めてください。


In [ ]:
CODE_COMMIT = "b96e4929783b32796fef0238834af85d72aceb9f"  # P5b実装を含む固定コミット（40桁）
EXPECTED_INPUT_SHA256 = "7b02611de118c11add0471744e64fc1b77cfb0aeb4fe285e4f29b9e7699d920a"  # bundleの報告値（64桁）
EXPECTED_REFERENCE_SHA256 = "02f01237752230937def6c9ffdecefbbab398d9d600a5ec890b7ad16a08b289f"  # bundleの報告値（64桁）
from pathlib import Path
import hashlib, json, os, re, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
BUNDLE = Path("/content/drive/MyDrive/viveMonte/p5/p5b")
RESULTS = BUNDLE / "colab_results"
for name, expected in (("p5b_inputs.npz", EXPECTED_INPUT_SHA256),
                       ("p5b_reference.npz", EXPECTED_REFERENCE_SHA256)):
    if not re.fullmatch(r"[0-9a-f]{64}", expected):
        raise ValueError("bundleの固定SHA-256を設定してください")
    actual = hashlib.sha256((BUNDLE/name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(name + " のSHA-256が違います。停止します")
    print(name, actual)
manifest = json.loads((BUNDLE/"p5b_manifest.json").read_text())
assert manifest["files"]["p5b_inputs.npz"]["sha256"] == EXPECTED_INPUT_SHA256
assert manifest["files"]["p5b_reference.npz"]["sha256"] == EXPECTED_REFERENCE_SHA256


In [ ]:
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("P5bを含む固定CODE_COMMITを設定してください")
REPO = Path("/content/p5b_checkout")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/gakugaku3333/chatcarlo.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == CODE_COMMIT
os.chdir(REPO)
os.environ["PYTHONPATH"] = str(REPO)
print("固定コミット", CODE_COMMIT)


## P5b: コンパイル確認（輸送は実行しません）
全カーネルの `RawKernel.compile()`、全文 SHA-256、NVRTC のログ全文と版を Drive に保存します。
エラーがあれば停止し、`p5b_compile.json` を Claude に渡してください。警告も確認対象です。


In [ ]:
subprocess.run([sys.executable, "-m", "prototypes.mv_electron.gpu.p5b_colab", "compile",
                "--bundle", str(BUNDLE), "--results", str(RESULTS)], check=True)
compiled = json.loads((RESULTS/"p5b_compile.json").read_text())
assert compiled["status"] == "pass"
assert compiled["source_sha256"] == manifest["source_sha256"]
print(json.dumps(compiled, indent=2, ensure_ascii=False))
print("ここで止め、コンパイル記録をClaudeに渡してください")


## P5c: 別の承認後のみ実行
C0→C1→C2→C3→C4→C5。不合格で停止し、種を変えません。速さは測りません。
C4 は各バッチを Drive に保存します。再開には同じ構成・同じ GPU の版を使ってください。
途中ファイルや不一致は削除せず停止します。


In [ ]:
P5C_APPROVED = False  # P5cの別承認を得た後だけTrueにする
if not P5C_APPROVED:
    raise RuntimeError("P5cは別の承認後に実行してください")
subprocess.run([sys.executable, "-m", "prototypes.mv_electron.gpu.p5b_colab", "run",
                "--bundle", str(BUNDLE), "--results", str(RESULTS)], check=True)
print((RESULTS/"p5c.json").read_text())
